# **`Prétraitement` des données pour Modélisation LLM**

## **Imports**

In [12]:
# Imports
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import random
import json

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

## **Chargement**

In [2]:
df = pd.read_csv('../data/medquad_clean.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (16359, 6)


,question,answer,source,focus_area,question_len,answer_len
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma,24,1850
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma,22,1209
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma,35,1607
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma,38,1873
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma,24,632


## **Split `Train/Val/Test` par Questions uniques**

In [3]:
# Obtenir les questions uniques
unique_questions = df['question'].unique()
print(f"Total questions uniques: {len(unique_questions)}")


Total questions uniques: 14979


In [6]:
# Split train (70%), temp (30%)
train_questions, temp_questions = train_test_split(
    unique_questions, test_size=0.3, random_state=RANDOM_STATE
)

# Split temp en validation (15%) et test (15%)
val_questions, test_questions = train_test_split(
    temp_questions, test_size=0.5, random_state=RANDOM_STATE
)

print(f"Train: {len(train_questions)} questions")
print(f"Validation: {len(val_questions)} questions")
print(f"Test: {len(test_questions)} questions")

Train: 10485 questions
Validation: 2247 questions
Test: 2247 questions


## **Création des datasets `Train/Val/Test` par Questions**

In [7]:
# Filtrer les lignes correspondantes
train_df = df[df['question'].isin(train_questions)]
val_df = df[df['question'].isin(val_questions)]
test_df = df[df['question'].isin(test_questions)]

print(f"Train examples: {len(train_df)}")
print(f"Validation examples: {len(val_df)}")
print(f"Test examples: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)}")

Train examples: 11494
Validation examples: 2431
Test examples: 2434
Total: 16359


## **Vérification d'absence de fuite**

In [8]:
# Vérifier qu'aucune question n'est partagée entre les ensembles
train_questions_set = set(train_df['question'])
val_questions_set = set(val_df['question'])
test_questions_set = set(test_df['question'])

print("Intersections:")
print(f"Train ∩ Val: {len(train_questions_set & val_questions_set)}")
print(f"Train ∩ Test: {len(train_questions_set & test_questions_set)}")
print(f"Val ∩ Test: {len(val_questions_set & test_questions_set)}")

Intersections:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


**Observation** : Split propre, aucune question commune entre les ensembles. Les données sont prêtes pour la suite sans risque de fuite.

## **Sauvegarde des splits**

In [10]:
# Sauvegarder les dataframes pour utilisation ultérieure
train_df.to_csv('../data/train.csv', index=False)
val_df.to_csv('../data/val.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

print("Fichiers sauvegardés:")
# print("- ../data/train.csv")
# print("- ../data/val.csv") 
# print("- ../data/test.csv")

# Vérification rapide
print(f"\nTaille des fichiers sauvegardés:")
print(f"Train: {len(train_df)} lignes")
print(f"Val: {len(val_df)} lignes")
print(f"Test: {len(test_df)} lignes")

Fichiers sauvegardés:

Taille des fichiers sauvegardés:
Train: 11494 lignes
Val: 2431 lignes
Test: 2434 lignes


## **Vérification - distribution des sources dans les splits**

In [11]:
# Vérifier que la distribution des sources est similaire entre les splits
print("Distribution des sources - Train:")
print(train_df['source'].value_counts(normalize=True).head())
print("\nDistribution des sources - Validation:")
print(val_df['source'].value_counts(normalize=True).head())
print("\nDistribution des sources - Test:")
print(test_df['source'].value_counts(normalize=True).head())

Distribution des sources - Train:
source
GHR                  0.331912
GARD                 0.327040
NIDDK                0.072038
NINDS                0.065773
MPlusHealthTopics    0.060118
Name: proportion, dtype: float64

Distribution des sources - Validation:
source
GARD                 0.338955
GHR                  0.335664
NINDS                0.068285
MPlusHealthTopics    0.058001
NIDDK                0.057589
Name: proportion, dtype: float64

Distribution des sources - Test:
source
GARD                 0.331142
GHR                  0.328266
NIDDK                0.072309
NINDS                0.068200
MPlusHealthTopics    0.061216
Name: proportion, dtype: float64


**Observation :** Distributions homogènes entre train/val/test (~33% GHR, ~33% GARD, etc.). Bon split, représentatif de l'ensemble.

## **Sauvegarde du `preprocessing`**

In [13]:
split_info = {
    'train_questions': list(train_questions),
    'val_questions': list(val_questions),
    'test_questions': list(test_questions),
    'RANDOM_STATE': RANDOM_STATE,
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df)
}

with open('../data/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print("Split info sauvegardé dans ../data/split_info.json")

Split info sauvegardé dans ../data/split_info.json


## **Résumé général**

Le dataset MedQuAD est passé de 16412 à 16359 exemples après suppression des doublons parfaits et des réponses manquantes. Les 14979 questions uniques ont été splittées en 70% train, 15% validation et 15% test, sans aucun chevauchement entre les ensembles. Les distributions des sources sont homogènes dans chaque split. Les fichiers train.csv, val.csv, test.csv et split_info.json sont sauvegardés.